<a href="https://colab.research.google.com/github/janpfrang-hash/distance-measurement-sensor/blob/main/Analysis_speed_fatigue_tester.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title
# 1. Install necessary libraries
!pip install ipyfilechooser -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks, savgol_filter
from scipy.interpolate import interp1d
import ipywidgets as widgets
from IPython.display import display
from google.colab import drive
from ipyfilechooser import FileChooser

# 2. Mount Google Drive
drive.mount('/content/drive')

print("\n--- Kinematic Profiler: Position, Velocity, Acceleration, Jerk ---")

# 3. Create the UI widgets
fc = FileChooser('/content/drive/MyDrive')
fc.title = '<b>1. Select Logfile (.txt, .csv)</b>'
fc.filter_pattern = ['*.txt', '*.csv']

start_time_input = widgets.FloatText(
    value=0.0,
    description='Start Time (s):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='150px')
)

cycles_input = widgets.IntText(
    value=20,
    description='Cycles to Plot:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='150px')
)

filter_window_input = widgets.IntText(
    value=51, # Increased default for better higher-order derivatives
    step=2,   # Must be an odd number
    description='Smoothing Window (odd #):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='220px')
)

process_button = widgets.Button(
    description="Generate Kinematic Plots",
    button_style='success',
    icon='line-chart',
    layout=widgets.Layout(width='300px', height='40px')
)
output = widgets.Output()

ui_box = widgets.VBox([
    fc,
    widgets.HTML("<b>2. Plot Settings</b>"),
    widgets.HBox([start_time_input, cycles_input, filter_window_input]),
    widgets.HTML("<i>Note: A larger smoothing window reduces derivative noise but may slightly round off sharp peaks.</i><br>"),
    process_button
])

display(ui_box, output)

def analyze_kinematics(b):
    with output:
        output.clear_output()

        filepath = fc.selected
        start_time = start_time_input.value
        target_cycles = cycles_input.value
        window_len = filter_window_input.value

        # Savitzky-Golay window must be odd
        if window_len % 2 == 0:
            window_len += 1
            print(f"⚠️ Smoothing window must be an odd number. Automatically adjusted to {window_len}.")

        if not filepath:
            print("⚠️ Please select a file from the list first!")
            return

        print(f"Loading data from: {filepath}...")

        try:
            # Load Zeit_s and Weg_mm
            df = pd.read_csv(filepath, sep=';', usecols=['Zeit_s', 'Weg_mm'])

            # 1. Clean data: Drop strict duplicates in time to allow interpolation
            df = df.dropna()
            df = df.drop_duplicates(subset=['Zeit_s']).reset_index(drop=True)

            # Apply User Start Time
            df = df[df['Zeit_s'] >= start_time].reset_index(drop=True)

            if df.empty:
                print(f"❌ No data found after Start Time: {start_time} s. Check your logfile timestamps.")
                return

        except Exception as e:
            print(f"❌ Error loading file. Details: {e}")
            return

        print("Interpolating to a strictly uniform time grid...")

        t_raw = df['Zeit_s'].values
        x_raw = df['Weg_mm'].values

        # Calculate the median time step to represent the true sampling rate (~200 Hz)
        dt = np.median(np.diff(t_raw))
        if dt <= 0:
            dt = 0.005 # Safe fallback to 200 Hz

        # Create a perfectly uniform time array
        t_uniform = np.arange(t_raw[0], t_raw[-1], dt)

        # Interpolate the raw position data onto the uniform time array
        f_interp = interp1d(t_raw, x_raw, kind='linear')
        x_uniform = f_interp(t_uniform)

        print("Detecting cycles...")

        # Find peaks in the uniform displacement data
        peaks, _ = find_peaks(x_uniform, prominence=0.5)

        if len(peaks) == 0:
            print("❌ No clear cycles detected. The amplitude might be too small.")
            return

        # Determine the cutoff index for the requested number of cycles
        if len(peaks) > target_cycles:
            cutoff_idx = peaks[target_cycles] + int((peaks[1] - peaks[0]) / 2)
            t = t_uniform[:cutoff_idx]
            x = x_uniform[:cutoff_idx]
            print(f"Successfully sliced {target_cycles} cycles.")
        else:
            t = t_uniform
            x = x_uniform
            print(f"⚠️ Only found {len(peaks)} cycles after {start_time} s. Plotting all of them.")

        print("Calculating derivatives & filtering noise...")

        # 1. Smooth Position
        x_smooth = savgol_filter(x, window_length=window_len, polyorder=3)

        # 2. Velocity (v = dx/dt) -> using the strict mathematical uniform 'dt'
        v = np.gradient(x_smooth, dt)
        v_smooth = savgol_filter(v, window_length=window_len, polyorder=3)

        # 3. Acceleration (a = dv/dt)
        a = np.gradient(v_smooth, dt)
        a_smooth = savgol_filter(a, window_length=window_len, polyorder=3)

        # 4. Jerk (j = da/dt)
        j = np.gradient(a_smooth, dt)
        j_smooth = savgol_filter(j, window_length=window_len, polyorder=3)

        print("Generating plots...")

        fig, axs = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
        fig.suptitle(f'System Kinematics ({len(peaks[:target_cycles])} Cycles from {t[0]:.1f}s)', fontsize=16, y=0.98)

        # --- Position Plot ---
        axs[0].plot(t, x, color='lightgray', label='Interpolated Raw Data', alpha=0.5)
        axs[0].plot(t, x_smooth, color='black', linewidth=2, label='Filtered Position')
        axs[0].set_ylabel('Position (mm)', fontsize=12, fontweight='bold')
        axs[0].legend(loc='upper right')
        axs[0].grid(True, linestyle='--', alpha=0.7)

        # --- Velocity Plot ---
        axs[1].plot(t, v_smooth, color='royalblue', linewidth=2)
        axs[1].set_ylabel('Velocity (mm/s)', fontsize=12, fontweight='bold', color='royalblue')
        axs[1].tick_params(axis='y', labelcolor='royalblue')
        axs[1].grid(True, linestyle='--', alpha=0.7)
        axs[1].fill_between(t, v_smooth, 0, color='royalblue', alpha=0.1)

        # --- Acceleration Plot ---
        axs[2].plot(t, a_smooth, color='darkorange', linewidth=2)
        axs[2].set_ylabel('Accel (mm/s²)', fontsize=12, fontweight='bold', color='darkorange')
        axs[2].tick_params(axis='y', labelcolor='darkorange')
        axs[2].grid(True, linestyle='--', alpha=0.7)

        # --- Jerk Plot ---
        axs[3].plot(t, j_smooth, color='firebrick', linewidth=1.5)
        axs[3].set_ylabel('Jerk (mm/s³)', fontsize=12, fontweight='bold', color='firebrick')
        axs[3].tick_params(axis='y', labelcolor='firebrick')
        axs[3].set_xlabel('Time (Seconds)', fontsize=12, fontweight='bold')
        axs[3].grid(True, linestyle='--', alpha=0.7)

        # Formatting limits and layout
        axs[3].set_xlim(t[0], t[-1])
        plt.tight_layout()
        plt.subplots_adjust(top=0.94)

        plt.show()
        print("Analysis complete!")

# Link button click to the function
process_button.on_click(analyze_kinematics)